<a href="https://colab.research.google.com/github/esmajasa/MLFlyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/esmajasa/MLFlyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

PREP

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

In [2]:
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add it in the "
        "Colab Secrets panel and enable notebook access."
    )

con = duckdb.connect()

# Escape quote characters without printing or storing the token
# in the notebook itself.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

del safe_token

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

print("Connected to the March 2026 warehouse partition.")

Connected to the March 2026 warehouse partition.


In [3]:
page_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- --------------------------------------------
        -- Feature window: March 1–21
        -- --------------------------------------------

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS feature_impressions,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS feature_clicks,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                 AND gsc_avg_position IS NOT NULL
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN report_date <= DATE '2026-03-21'
                     AND gsc_avg_position IS NOT NULL
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                 AND gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_position_volatility,

        COUNT(*) FILTER (
            WHERE report_date <= DATE '2026-03-21'
              AND gsc_impressions > 0
        ) AS feature_active_days,

        -- --------------------------------------------
        -- Outcome window: March 22–31
        -- --------------------------------------------

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS target_impressions,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_clicks
                ELSE 0
            END
        ) AS target_clicks,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                 AND gsc_avg_position IS NOT NULL
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN report_date >= DATE '2026-03-22'
                     AND gsc_avg_position IS NOT NULL
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS target_avg_position

    FROM {MARCH_DAILY}

    WHERE ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-21'
                THEN gsc_impressions
                ELSE 0
            END
        ) >= 100

        AND

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-22'
                THEN gsc_impressions
                ELSE 0
            END
        ) >= 50
""").df()

print(
    "Page-level rows after aggregation and filters:",
    f"{len(page_frame):,}",
)

display(page_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows after aggregation and filters: 17,423


,client_hash_id,content_hash_id,feature_impressions,feature_clicks,feature_avg_position,feature_position_volatility,feature_active_days,target_impressions,target_clicks,target_avg_position
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,379.0,2.0,4.044855,0.851947,7,79.0,0.0,5.746835
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3013.0,17.0,4.144374,0.797423,17,930.0,6.0,4.766667
2,client_65de48885f4ef01b,content_3c286ded8bd68120,1241.0,10.0,8.983078,1.243600,12,939.0,5.0,7.673056
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,366.0,2.0,5.614754,1.545314,10,137.0,6.0,6.029197
4,client_65de48885f4ef01b,content_d6c71358297cfd6a,152.0,5.0,3.309211,5.235360,5,517.0,17.0,2.651838


In [4]:
# Past CTR: an honest feature
page_frame["feature_ctr"] = (
    100
    * page_frame["feature_clicks"]
    / page_frame["feature_impressions"]
)

# Compress the highly skewed impression counts
page_frame["log_feature_impressions"] = np.log1p(
    page_frame["feature_impressions"]
)

# Later CTR: used to construct the outcome, not as a feature
page_frame["target_ctr"] = (
    100
    * page_frame["target_clicks"]
    / page_frame["target_impressions"]
)

FEATURES = [
    "feature_ctr",
    "log_feature_impressions",
    "feature_avg_position",
    "feature_position_volatility",
    "feature_active_days",
]

assert len(FEATURES) == 5

print("The five honest features are:")

for feature in FEATURES:
    print("-", feature)

display(page_frame[FEATURES].head())

The five honest features are:
- feature_ctr
- log_feature_impressions
- feature_avg_position
- feature_position_volatility
- feature_active_days


,feature_ctr,log_feature_impressions,feature_avg_position,feature_position_volatility,feature_active_days
0,0.527704,5.940171,4.044855,0.851947,7
1,0.564222,8.011023,4.144374,0.797423,17
2,0.805802,7.124478,8.983078,1.243600,12
3,0.546448,5.905362,5.614754,1.545314,10
4,3.289474,5.030438,3.309211,5.235360,5


In [5]:
position_edges = [
    0,
    3,
    10,
    20,
    50,
    np.inf,
]

position_names = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep",
]

page_frame["target_position_band"] = pd.cut(
    page_frame["target_avg_position"],
    bins=position_edges,
    labels=position_names,
)

display(
    page_frame[
        [
            "target_avg_position",
            "target_position_band",
            "target_ctr",
        ]
    ].head()
)

,target_avg_position,target_position_band,target_ctr
0,5.746835,page_1,0.000000
1,4.766667,page_1,0.645161
2,7.673056,page_1,0.532481
3,6.029197,page_1,4.379562
4,2.651838,top_3,3.288201


In [6]:
band_median_target_ctr = (
    page_frame
    .groupby(
        "target_position_band",
        observed=True,
    )["target_ctr"]
    .transform("median")
)

page_frame["future_low_ctr_opportunity"] = (
    page_frame["target_ctr"]
    < band_median_target_ctr
).astype(int)

label_summary = (
    page_frame["future_low_ctr_opportunity"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="pages")
)

display(label_summary)

print(
    "Positive-label share:",
    round(
        page_frame[
            "future_low_ctr_opportunity"
        ].mean(),
        3,
    ),
)

,label,pages
0,0,8769
1,1,8654


Positive-label share: 0.497


In [7]:
REQUIRED_COLUMNS = [
    "client_hash_id",
    "content_hash_id",
    "feature_ctr",
    "log_feature_impressions",
    "feature_avg_position",
    "feature_position_volatility",
    "feature_active_days",
    "target_impressions",
    "target_clicks",
    "target_ctr",
    "target_avg_position",
    "target_position_band",
    "future_low_ctr_opportunity",
]

missing_required_columns = (
    set(REQUIRED_COLUMNS)
    - set(page_frame.columns)
)

print(
    "Missing required columns:",
    missing_required_columns,
)

assert not missing_required_columns

Missing required columns: set()


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
FEATURES = [
    "feature_ctr",
    "log_feature_impressions",
    "feature_avg_position",
    "feature_position_volatility",
    "feature_active_days",
]

LABEL = "future_low_ctr_opportunity"

CONTEXT = [
    "client_hash_id",
    "content_hash_id",
]

assert len(FEATURES) == 5
assert LABEL not in FEATURES
assert not set(CONTEXT).intersection(FEATURES)

X_raw = page_frame[FEATURES].copy()
y = page_frame[LABEL].copy()
groups = page_frame["client_hash_id"].copy()

print("Feature-vector shape:", X_raw.shape)
print("Number of features:", len(FEATURES))
print("Label rows:", len(y))
print("Pseudonymized clients:", groups.nunique())

display(X_raw.head())

Feature-vector shape: (17423, 5)
Number of features: 5
Label rows: 17423
Pseudonymized clients: 25


,feature_ctr,log_feature_impressions,feature_avg_position,feature_position_volatility,feature_active_days
0,0.527704,5.940171,4.044855,0.851947,7
1,0.564222,8.011023,4.144374,0.797423,17
2,0.805802,7.124478,8.983078,1.243600,12
3,0.546448,5.905362,5.614754,1.545314,10
4,3.289474,5.030438,3.309211,5.235360,5


In [9]:
missing_summary = pd.DataFrame({
    "dtype": X_raw.dtypes.astype(str),
    "missing_count": X_raw.isna().sum(),
    "missing_pct": (
        X_raw.isna().mean() * 100
    ).round(2),
})

display(missing_summary)

,dtype,missing_count,missing_pct
feature_ctr,float64,0,0.00
log_feature_impressions,float64,0,0.00
feature_avg_position,float64,0,0.00
feature_position_volatility,float64,329,1.89
feature_active_days,int64,0,0.00


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

numeric_preprocessor = SimpleImputer(
    strategy="median"
)

print(
    "All five features are numeric. "
    "No categorical encoding is required."
)

All five features are numeric. No categorical encoding is required.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing-value handling | Categorical? | Available before March 22? |
|---|---|---|---|---|
| `feature_ctr` | Clicks divided by impressions during March 1–21, multiplied by 100 | Not expected after the minimum-impression filter; otherwise median-imputed in the training pipeline | No | Yes—its window ends March 21 |
| `log_feature_impressions` | Log-transformed total impressions during March 1–21 | Not expected; otherwise median-imputed in the training pipeline | No | Yes—its window ends March 21 |
| `feature_avg_position` | Impression-weighted average search position during March 1–21 | Checked explicitly; median-imputed only if missing | No | Yes—uses no later position data |
| `feature_position_volatility` | Standard deviation of daily search position during March 1–21 | May be missing when too few positions exist; median-imputed in the training pipeline | No | Yes—uses only March 1–21 |
| `feature_active_days` | Number of March 1–21 days with at least one impression | Not expected; otherwise median-imputed | No | Yes—count stops March 21 |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
#check for forbidden columns
FORBIDDEN_FEATURES = {
    # Outcome-window measurements
    "target_impressions",
    "target_clicks",
    "target_ctr",
    "target_avg_position",
    "target_position_band",

    # The label itself
    "future_low_ctr_opportunity",

    # Identifiers
    "client_hash_id",
    "content_hash_id",

    # Existing-product decisions
    "health_score",
    "priority_score",
    "needs_ctr_fix",
    "is_quick_win",
    "action_type",
}

forbidden_overlap = (
    set(FEATURES) & FORBIDDEN_FEATURES
)

print(
    "Forbidden fields in honest features:",
    forbidden_overlap,
)

assert not forbidden_overlap, (
    f"Unsafe features found: {forbidden_overlap}"
)

Forbidden fields in honest features: set()


In [12]:
#verify feature timeline
timing_check = pd.DataFrame({
    "feature": FEATURES,
    "window_start": ["2026-03-01"] * len(FEATURES),
    "window_end": ["2026-03-21"] * len(FEATURES),
    "decision_date": ["2026-03-22"] * len(FEATURES),
    "label_window_start": ["2026-03-22"] * len(FEATURES),
    "label_window_end": ["2026-03-31"] * len(FEATURES),
})

timing_check["available_before_decision"] = (
    pd.to_datetime(timing_check["window_end"])
    < pd.to_datetime(timing_check["decision_date"])
)

display(timing_check)

assert timing_check[
    "available_before_decision"
].all()

,feature,window_start,window_end,decision_date,label_window_start,label_window_end,available_before_decision
0,feature_ctr,2026-03-01,2026-03-21,2026-03-22,2026-03-22,2026-03-31,True
1,log_feature_impressions,2026-03-01,2026-03-21,2026-03-22,2026-03-22,2026-03-31,True
2,feature_avg_position,2026-03-01,2026-03-21,2026-03-22,2026-03-22,2026-03-31,True
3,feature_position_volatility,2026-03-01,2026-03-21,2026-03-22,2026-03-22,2026-03-31,True
4,feature_active_days,2026-03-01,2026-03-21,2026-03-22,2026-03-22,2026-03-31,True


In [13]:
#print the base rate
positive_rate = y.mean()
majority_baseline = max(
    positive_rate,
    1 - positive_rate,
)

print(
    "Positive-label share:",
    round(positive_rate, 3),
)

print(
    "Majority-class accuracy baseline:",
    round(majority_baseline, 3),
)

Positive-label share: 0.497
Majority-class accuracy baseline: 0.503


In [14]:
#define the quick model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)

def fit_and_score(
    feature_columns,
    train_index,
    test_index,
):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=150,
            random_state=42,
            n_jobs=-1,
        ),
    )

    model.fit(
        page_frame.iloc[train_index][feature_columns],
        y.iloc[train_index],
    )

    predictions = model.predict(
        page_frame.iloc[test_index][feature_columns]
    )

    score = balanced_accuracy_score(
        y.iloc[test_index],
        predictions,
    )

    return score

In [15]:
#honest random split
row_indices = np.arange(len(page_frame))

random_train, random_test = train_test_split(
    row_indices,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

random_honest_score = fit_and_score(
    FEATURES,
    random_train,
    random_test,
)

print(
    "Honest random-split balanced accuracy:",
    round(random_honest_score, 3),
)

Honest random-split balanced accuracy: 0.713


In [16]:
#honest client-grouped split
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

group_train, group_test = next(
    group_splitter.split(
        X_raw,
        y,
        groups=groups,
    )
)

grouped_honest_score = fit_and_score(
    FEATURES,
    group_train,
    group_test,
)

train_clients = set(groups.iloc[group_train])
test_clients = set(groups.iloc[group_test])

assert train_clients.isdisjoint(test_clients)

print(
    "Honest client-grouped balanced accuracy:",
    round(grouped_honest_score, 3),
)

print(
    "Clients shared between train and test:",
    len(train_clients & test_clients),
)

Honest client-grouped balanced accuracy: 0.675
Clients shared between train and test: 0


In [17]:
#deliberately adding the lead
LEAKY_FEATURES = FEATURES + ["target_ctr"]

leaky_random_score = fit_and_score(
    LEAKY_FEATURES,
    random_train,
    random_test,
)

print(
    "Honest random-split score:",
    round(random_honest_score, 3),
)

print(
    "Leaky score with target_ctr:",
    round(leaky_random_score, 3),
)

print(
    "Artificial score increase:",
    round(
        leaky_random_score
        - random_honest_score,
        3,
    ),
)

Honest random-split score: 0.713
Leaky score with target_ctr: 0.95
Artificial score increase: 0.237


In [18]:
#removing the leak
FINAL_FEATURES = FEATURES.copy()

assert "target_ctr" not in FINAL_FEATURES
assert not (
    set(FINAL_FEATURES)
    & FORBIDDEN_FEATURES
)

honest_feature_frame = page_frame[
    CONTEXT + FINAL_FEATURES + [LABEL]
].copy()

print(
    "Final honest features:",
    FINAL_FEATURES,
)

print(
    "target_ctr retained as feature:",
    "target_ctr" in FINAL_FEATURES,
)

Final honest features: ['feature_ctr', 'log_feature_impressions', 'feature_avg_position', 'feature_position_volatility', 'feature_active_days']
target_ctr retained as feature: False


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Reason |
|---|---|
| `target_ctr` | Comes from March 22–31 and directly constructs the low-CTR proxy |
| `target_clicks` | Used to calculate `target_ctr` and comes from the outcome period |
| `target_impressions` | Comes from the outcome period and participates in the proxy calculation |
| `target_avg_position` | Measured after the decision moment |
| `target_position_band` | Derived from later position and used to define the proxy |
| `future_low_ctr_opportunity` | This is the label, not an input feature |
| `client_hash_id` | Used for client-grouped validation; the model must not memorize client identity |
| `content_hash_id` | Pseudonymized identifier with no meaningful numeric interpretation |
| Product scores and flags | Encode decisions made by the existing system and would create a circular result |
| Raw client names, URLs and queries | Private and not part of the public-safe feature vector |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.